# Advanced Big Data Capstone: TaaSim Urban Mobility
## Week 4: Stream Processing II — Demand Aggregation (Flink Job 2)

**Objective:** Ingest normalized GPS streams (processed.gps) and trip requests (raw.trips), aggregate supply (active vehicles) and demand (pending requests) per zone every 30 seconds, compute supply/demand ratio, and sink to Cassandra demand_zones table and processed.demand Kafka topic.

### Key Architecture Points
- **Dual Kafka sources**: processed.gps (output from Job 1) + raw.trips (trip requests)
- **Event-time windowing**: 30-second tumbling windows per zone
- **Aggregation logic**: count_vehicles, count_requests, ratio = requests / max(vehicles, 1)
- **Sink 1**: Cassandra demand_zones (for Grafana heatmap)
- **Sink 2**: Kafka processed.demand (for downstream jobs)

In [1]:
# Cell 1: Environment Setup
import os, shutil, urllib.request, sys
from pathlib import Path
from pyflink.find_flink_home import _find_flink_home
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.datastream.connectors.kafka import FlinkKafkaConsumer, FlinkKafkaProducer
from pyflink.datastream.functions import AggregateFunction, WindowFunction
from pyflink.datastream.window import TumblingEventTimeWindow
from pyflink.common.serialization import SimpleStringSchema
from pyflink.common.typeinfo import Types
import json
from datetime import datetime
import time

# Toggle this to True only for short debugging sessions.
DEBUG_STREAM_OUTPUT = False

# Fix Java on Windows.
if os.name == "nt":
    for home in [
        r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot",
        r"C:\Program Files\Adoptium\jdk-17.0.7",
    ]:
        java_bin = Path(home) / "bin" / "java.exe"
        if java_bin.exists():
            os.environ["JAVA_HOME"] = str(Path(home))
            os.environ["PATH"] = f"{java_bin.parent}{os.pathsep}" + os.environ.get("PATH", "")
            break

java_cmd = shutil.which("java")
if not java_cmd:
    raise RuntimeError("Java not found. Install JDK 11/17.")
print(f"Using Java: {java_cmd}")

# Required jars: Kafka + Cassandra connectors (reuse from Job 1).
flink_lib = Path(_find_flink_home()) / "lib"
jars = {
    "flink-connector-kafka-3.0.2-1.18.jar":
        "https://repo1.maven.org/maven2/org/apache/flink/flink-connector-kafka/3.0.2-1.18/flink-connector-kafka-3.0.2-1.18.jar",
    "kafka-clients-3.2.3.jar":
        "https://repo1.maven.org/maven2/org/apache/kafka/kafka-clients/3.2.3/kafka-clients-3.2.3.jar",
    "flink-connector-cassandra_2.12-3.2.0-1.18.jar":
        "https://repo1.maven.org/maven2/org/apache/flink/flink-connector-cassandra_2.12/3.2.0-1.18/flink-connector-cassandra_2.12-3.2.0-1.18.jar",
}

for jar_name, url in jars.items():
    dest = flink_lib / jar_name
    if not dest.exists():
        print(f"Downloading {jar_name} ...")
        urllib.request.urlretrieve(url, dest)
        print(f"  -> Saved to {dest}")
    else:
        print(f"  OK: {jar_name} already present")

# Force Flink Python operators to use the same interpreter as this Jupyter kernel.
os.environ["PYFLINK_CLIENT_EXECUTABLE"] = sys.executable
os.environ["PYFLINK_PYTHON_EXECUTABLE"] = sys.executable

env = StreamExecutionEnvironment.get_execution_environment()
env.set_python_executable(sys.executable)
env.set_parallelism(1)
print(f"PyFlink worker python: {sys.executable}")
print("Flink environment initialized for Job 2.")

ImportError: cannot import name 'TumblingEventTimeWindow' from 'pyflink.datastream.window' (c:\Users\nmira\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyflink\datastream\window.py)

In [ ]:
# Cell 2: Dual Kafka Sources
kafka_props = {
    'bootstrap.servers': 'localhost:9094',
    'group.id': 'taasim-jupyter-job2-demand',
    'auto.offset.reset': 'earliest'
}

# Source 1: processed.gps (vehicle positions from Job 1)
gps_consumer = FlinkKafkaConsumer(
    topics='processed.gps',
    deserialization_schema=SimpleStringSchema(),
    properties=kafka_props
)
gps_stream = env.add_source(gps_consumer)

# Source 2: raw.trips (trip requests from producer)
trips_consumer = FlinkKafkaConsumer(
    topics='raw.trips',
    deserialization_schema=SimpleStringSchema(),
    properties=kafka_props
)
trips_stream = env.add_source(trips_consumer)

print("✅ Dual Kafka sources configured.")
print("   - processed.gps (vehicles from Job 1)")
print("   - raw.trips (trip requests)")

In [ ]:
# Cell 3: Parse and Extract Streams

def parse_gps_json(data_string):
    """Parse processed.gps event: extract zone_id, taxi_id, event_time"""
    try:
        data = json.loads(data_string)
        return (
            "vehicle",  # type marker
            int(data['zone_id']),  # zone_id
            int(data['event_time']),  # event_time (seconds)
            int(data['taxi_id']),  # taxi_id (for deduplication)
            str(data.get('status', 'UNKNOWN'))  # vehicle status
        )
    except Exception:
        return None

def parse_trips_json(data_string):
    """Parse raw.trips event: extract zone_id, trip_id, event_time"""
    try:
        data = json.loads(data_string)
        return (
            "trip",  # type marker
            int(data['origin_zone']),  # zone_id (origin for demand)
            int(data['requested_at']),  # event_time (seconds)
            str(data['trip_id']),  # trip_id (for deduplication)
            "PENDING"  # trip status
        )
    except Exception:
        return None

# Parse GPS stream
gps_parsed = gps_stream.map(
    parse_gps_json,
    output_type=Types.TUPLE([
        Types.STRING(), Types.INT(), Types.LONG(), Types.STRING(), Types.STRING()
    ])
).filter(lambda x: x is not None)

# Parse trips stream
trips_parsed = trips_stream.map(
    parse_trips_json,
    output_type=Types.TUPLE([
        Types.STRING(), Types.INT(), Types.LONG(), Types.STRING(), Types.STRING()
    ])
).filter(lambda x: x is not None)

if DEBUG_STREAM_OUTPUT:
    gps_parsed.print()
    trips_parsed.print()
    print("Debug sinks attached.")
else:
    print("Debug sinks disabled.")

print("✅ Both streams parsed into (type, zone_id, event_time, id, status).")

In [ ]:
# Cell 4: Union Streams and Assign Watermarks
from pyflink.common.watermark_strategy import WatermarkStrategy, TimestampAssigner
from pyflink.common.time import Duration

class EventTimeExtractor(TimestampAssigner):
    """Extract event_time from tuple[1] (zone_id) field and convert to milliseconds"""
    def extract_timestamp(self, value, record_timestamp):
        # value = (type, zone_id, event_time, id, status)
        # Extract event_time (field[2]) and convert to milliseconds
        return int(value[2] * 1000)

# Create watermark strategy: 3-minute allowed lateness (same as Job 1)
watermark_strategy = WatermarkStrategy \
    .for_bounded_out_of_orderness(Duration.of_minutes(3)) \
    .with_timestamp_assigner(EventTimeExtractor())

# Union both parsed streams
combined_stream = gps_parsed.union(trips_parsed)

# Assign timestamps and watermarks to combined stream
event_time_stream = combined_stream.assign_timestamps_and_watermarks(watermark_strategy)

if DEBUG_STREAM_OUTPUT:
    event_time_stream.print()
    print("Debug sink attached for event_time_stream.")
else:
    print("Debug sink disabled for event_time_stream.")

print("✅ Streams united and watermarks assigned (3-min allowed lateness).")

In [ ]:
# Cell 5: Key by zone_id for Windowing Parallelization

def extract_zone_id(record):
    """Extract zone_id from combined stream tuple"""
    return int(record[1])

# Keyed stream by zone_id to parallelize windowing per zone
keyed_stream = event_time_stream.key_by(
    lambda record: int(record[1])  # zone_id is field[1]
)

print("✅ Stream keyed by zone_id for per-zone aggregation.")

In [ ]:
# Cell 6: Apply 30-Second Tumbling Window

# Apply 30-second tumbling event-time window per zone
from pyflink.datastream.window import TimeWindow

windowed_stream = keyed_stream.window(TumblingEventTimeWindow.of(Duration.of_seconds(30)))

print("✅ 30-second tumbling window applied per zone_id.")
print("   Each window covers 30 seconds of event time.")

In [ ]:
# Cell 7: Aggregation Logic - Count Vehicles & Requests per Zone per Window

class DemandAccumulator:
    """Accumulator for window aggregation: tracks vehicles, requests, and timestamps"""
    def __init__(self):
        self.vehicle_ids = set()  # Distinct vehicles in window
        self.trip_ids = set()     # Distinct trips in window
        self.zone_id = None
        self.window_start = None

class DemandAggregateFunction(AggregateFunction):
    """
    Aggregate function: count distinct vehicles + distinct requests per zone per 30-sec window
    """
    
    def create_accumulator(self):
        return DemandAccumulator()
    
    def add(self, value, accumulator):
        # value = (type, zone_id, event_time, id, status)
        event_type = value[0]
        zone_id = int(value[1])
        entity_id = value[3]
        
        accumulator.zone_id = zone_id
        
        if event_type == "vehicle":
            accumulator.vehicle_ids.add(entity_id)
        elif event_type == "trip":
            accumulator.trip_ids.add(entity_id)
        
        return accumulator
    
    def get_result(self, accumulator):
        # Return intermediate result tuple
        vehicle_count = len(accumulator.vehicle_ids)
        trip_count = len(accumulator.trip_ids)
        ratio = trip_count / max(vehicle_count, 1)  # Prevent division by zero
        
        return (
            accumulator.zone_id,
            vehicle_count,
            trip_count,
            float(ratio)
        )
    
    def merge(self, acc_a, acc_b):
        # Merge two accumulators (for parallel execution)
        acc_a.vehicle_ids.update(acc_b.vehicle_ids)
        acc_a.trip_ids.update(acc_b.trip_ids)
        return acc_a

# Apply aggregate function to windowed stream
aggregated_stream = windowed_stream.aggregate(
    DemandAggregateFunction(),
    output_type=Types.TUPLE([
        Types.INT(), Types.INT(), Types.INT(), Types.DOUBLE()
    ])
)

if DEBUG_STREAM_OUTPUT:
    aggregated_stream.print()
    print("Debug sink attached for aggregated_stream.")
else:
    print("Debug sink disabled for aggregated_stream.")

print("✅ Aggregation configured:")
print("   - Count distinct vehicles per zone per 30-sec window")
print("   - Count distinct trip requests per zone per 30-sec window")
print("   - Compute ratio = requests / max(vehicles, 1)")

In [ ]:
# Cell 8: Add Window Context and Format for Sinks

class DemandWindowFunction(WindowFunction):
    """Window function to add window_start timestamp to aggregation result"""
    
    def apply(self, key, window, inputs):
        # inputs is an iterable of aggregation results (should have 1 element)
        result = None
        for agg_result in inputs:
            result = agg_result
            break  # Take the first (only) result
        
        if result:
            zone_id = result[0]
            vehicle_count = result[1]
            trip_count = result[2]
            ratio = result[3]
            window_start = window.start  # Get window start time in milliseconds
            
            # Return final record: (zone_id, vehicle_count, trip_count, ratio, window_start_ms)
            yield (
                zone_id,
                vehicle_count,
                trip_count,
                ratio,
                int(window_start)
            )

# Apply window function to add window context
windowed_aggregated = aggregated_stream.apply_window_function(
    DemandWindowFunction(),
    output_type=Types.TUPLE([
        Types.INT(), Types.INT(), Types.INT(), Types.DOUBLE(), Types.LONG()
    ])
)

if DEBUG_STREAM_OUTPUT:
    windowed_aggregated.print()
    print("Debug sink attached for windowed_aggregated.")
else:
    print("Debug sink disabled for windowed_aggregated.")

print("✅ Window context added (zone_id, active_vehicles, pending_requests, ratio, window_start_ms).")

In [ ]:
# Cell 9: Cassandra Sink - Write to demand_zones table

from pyflink.datastream.connectors.cassandra import CassandraSink

cassandra_query = (
    "INSERT INTO taasim.demand_zones "
    "(city, zone_id, window_start, active_vehicles, pending_requests, ratio) "
    "VALUES (?, ?, ?, ?, ?, ?);"
)

cassandra_sink = (
    CassandraSink.add_sink(windowed_aggregated)
    .set_host("localhost", 9042)
    .set_query(cassandra_query)
    .build()
)

# Transform before Cassandra sink: add city and rearrange for query parameters
def format_for_cassandra(record):
    # record = (zone_id, active_vehicles, pending_requests, ratio, window_start_ms)
    zone_id = record[0]
    vehicle_count = record[1]
    trip_count = record[2]
    ratio = record[3]
    window_start = record[4]
    
    # Return tuple matching query parameter order: (city, zone_id, window_start, vehicles, requests, ratio)
    return (
        "Casablanca",
        int(zone_id),
        int(window_start),
        int(vehicle_count),
        int(trip_count),
        float(ratio)
    )

# Apply format transformation
cassandra_stream = windowed_aggregated.map(
    format_for_cassandra,
    output_type=Types.TUPLE([
        Types.STRING(), Types.INT(), Types.LONG(), Types.INT(), Types.INT(), Types.DOUBLE()
    ])
)

# Add Cassandra sink
cassandra_sink = (
    CassandraSink.add_sink(cassandra_stream)
    .set_host("localhost", 9042)
    .set_query(cassandra_query)
    .build()
)

print("✅ Cassandra sink configured for demand_zones table.")

In [ ]:
# Cell 10: Kafka Sink - Write to processed.demand Topic

def tuple_to_demand_json(record):
    """Convert aggregation tuple to JSON for processed.demand topic"""
    # record = (zone_id, active_vehicles, pending_requests, ratio, window_start_ms)
    zone_id = record[0]
    vehicle_count = record[1]
    trip_count = record[2]
    ratio = record[3]
    window_start = record[4]
    
    payload = {
        "schema_version": "1.0.0",
        "event_type": "DEMAND_AGGREGATED",
        "zone_id": int(zone_id),
        "active_vehicles": int(vehicle_count),
        "pending_requests": int(trip_count),
        "supply_demand_ratio": float(ratio),
        "window_start_ms": int(window_start),
        "ingested_at": int(time.time() * 1000),
        "city": "Casablanca"
    }
    return json.dumps(payload)

# Transform for Kafka sink
kafka_stream = windowed_aggregated.map(
    tuple_to_demand_json,
    output_type=Types.STRING()
)

kafka_producer_props = {
    "bootstrap.servers": "localhost:9094",
    "transaction.timeout.ms": "900000",
}

kafka_sink = FlinkKafkaProducer(
    topic="processed.demand",
    serialization_schema=SimpleStringSchema(),
    producer_config=kafka_producer_props,
)

kafka_stream.add_sink(kafka_sink)

print("✅ Kafka sink configured for processed.demand topic.")

In [ ]:
# Cell 11: Job Status Check

if "job_client" in globals():
    try:
        job_id = job_client.get_job_id()
        status = job_client.get_job_status().result()
        print(f"JobID: {job_id}")
        print(f"Job status: {status}")
        print("Job runs asynchronously in the notebook kernel.")
    except Exception as ex:
        print(f"Could not read job status: {ex}")
else:
    print("No job submitted yet. Run the final execution cell first.")

In [ ]:
# Cell 12: Execute Job 2

should_submit = True

if "job_client" in globals():
    try:
        current_status = job_client.get_job_status().result()
        print(f"Existing job status: {current_status}")
        print(f"Existing JobID: {job_client.get_job_id()}")

        if "RUNNING" in str(current_status):
            print("Job is already running. Skipping re-submit to avoid graph reset errors.")
            should_submit = False
        else:
            print("Existing job is not running. Submitting a fresh job...")
    except Exception as ex:
        print(f"Existing job client is stale/unavailable: {ex}")
        print("Submitting a fresh job...")

if should_submit:
    job_client = env.execute_async("Jupyter: Job 2 - Demand Aggregator (30-sec tumbling window)")
    print(f"Job submitted. JobID: {job_client.get_job_id()}")
    print("\n✅ Job 2 is running asynchronously!")
    print("   - Consuming processed.gps and raw.trips")
    print("   - Aggregating supply (vehicles) and demand (requests) per zone every 30s")
    print("   - Writing to Cassandra demand_zones table")
    print("   - Publishing to processed.demand Kafka topic")

## Week 4: Job 2 Architecture Summary

### Data Flow
```
processed.gps (Job 1 output)     raw.trips (Trip Requests)
     ↓                                 ↓
  Parse GPS                        Parse Trips
  (extract zone_id, taxi_id)      (extract zone_id, trip_id)
     ↓                                 ↓
  Union Streams + Watermarks (3-min lateness)
     ↓
  Key by zone_id (partition per zone)
     ↓
  30-Second Tumbling Window
     ↓
  Aggregate: count distinct vehicles + requests per zone
     ↓
     ├→ Cassandra: demand_zones (zone_id, window_start, vehicles, requests, ratio)
     └→ Kafka: processed.demand (JSON events for Job 3)
```

### Key Metrics Computed
- **active_vehicles**: Distinct taxi_ids seen in 30-sec window per zone
- **pending_requests**: Distinct trip_ids seen in 30-sec window per zone
- **supply_demand_ratio**: pending_requests / max(active_vehicles, 1)

### Next Steps
1. **Verify Job 2 is running** (check job status above)
2. **Query Cassandra** to see demand_zones table being populated:
   ```bash
   docker exec taasim-cassandra cqlsh -e \
     "SELECT * FROM taasim.demand_zones WHERE city = 'Casablanca' LIMIT 10;"
   ```
3. **Check processed.demand topic** for Kafka messages:
   ```bash
   docker exec taasim-kafka kafka-console-consumer.sh \
     --bootstrap-server localhost:9092 \
     --topic processed.demand --from-beginning --max-messages 5
   ```
4. **Next Week**: Implement Job 3 (Trip Matcher) to match requests to vehicles in real time